# Programming Question Retrieval

## 1. Import Libraries¶


In [1]:
!pip install beautifulsoup4

In [2]:
import pandas as pd
import numpy as np
import os
import re
from bs4 import BeautifulSoup

## 2. Load Data

In [3]:

train_df = pd.read_csv(
    "/kaggle/input/datasets/imoore/60k-stack-overflow-questions-with-quality-rate/train.csv"
)

valid_df = pd.read_csv(
    "/kaggle/input/datasets/imoore/60k-stack-overflow-questions-with-quality-rate/valid.csv"
)

print("Datasets loaded successfully!")


Datasets loaded successfully!



### 1.2 Exploring the Dataset

In this step, we explore the dataset by displaying sample records, examining column names, and checking the general structure of the training data.

This helps us understand the available information before starting the data cleaning process.


In [4]:
# Display first 5 rows
display(train_df.head())

display(valid_df.head())

# Display column names
print("Columns:")
print(train_df.columns.tolist())

# Dataset information
train_df.info()

,Id,Title,Body,Tags,CreationDate,Y
0,34552656,Java: Repeat Task Every Random Seconds,<p>I'm already familiar with repeating tasks e...,<java><repeat>,2016-01-01 00:21:59,LQ_CLOSE
1,34553034,Why are Java Optionals immutable?,<p>I'd like to understand why Java 8 Optionals...,<java><optional>,2016-01-01 02:03:20,HQ
2,34553174,Text Overlay Image with Darkened Opacity React...,<p>I am attempting to overlay a title over an ...,<javascript><image><overlay><react-native><opa...,2016-01-01 02:48:24,HQ
3,34553318,Why ternary operator in swift is so picky?,"<p>The question is very simple, but I just cou...",<swift><operators><whitespace><ternary-operato...,2016-01-01 03:30:17,HQ
4,34553755,hide/show fab with scale animation,<p>I'm using custom floatingactionmenu. I need...,<android><material-design><floating-action-but...,2016-01-01 05:21:48,HQ


,Id,Title,Body,Tags,CreationDate,Y
0,34552974,How to get all the child records from differen...,I am having 4 different tables like \r\nselect...,<sql><sql-server>,2016-01-01 01:44:52,LQ_EDIT
1,34554721,Retrieve all except some data of the another t...,I have two table m_master and tbl_appointment\...,<php><mysql><sql><codeigniter><mysqli>,2016-01-01 08:43:50,LQ_EDIT
2,34555135,Pandas: read_html,<p>I'm trying to extract US states from wiki U...,<python><pandas>,2016-01-01 09:55:22,HQ
3,34555448,Reader Always gimme NULL,"I'm so new to C#, I wanna make an application ...",<sql-server><c#-4.0>,2016-01-01 10:43:45,LQ_EDIT
4,34555752,php rearrange array elements based on condition,basically i have this array:\r\n\r\n array(...,<php>,2016-01-01 11:34:09,LQ_EDIT


Columns:
['Id', 'Title', 'Body', 'Tags', 'CreationDate', 'Y']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45000 entries, 0 to 44999
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Id            45000 non-null  int64 
 1   Title         45000 non-null  object
 2   Body          45000 non-null  object
 3   Tags          45000 non-null  object
 4   CreationDate  45000 non-null  object
 5   Y             45000 non-null  object
dtypes: int64(1), object(5)
memory usage: 2.1+ MB



### 1.3 Dataset Dimensions

In this step, we examine the dimensions of both the training and validation datasets.

We identify the total number of records and columns in each dataset to understand the overall dataset size.


In [5]:
print("Training Dataset Shape:", train_df.shape)
print("Validation Dataset Shape:", valid_df.shape)

print("\nTotal Number of Records:",
      len(train_df) + len(valid_df))

Training Dataset Shape: (45000, 6)
Validation Dataset Shape: (15000, 6)

Total Number of Records: 60000



### 1.4 Missing Values Analysis

In this step, we check for missing values in both datasets.

Identifying missing values is important because incomplete records may affect the quality of the document collection and the retrieval process.


In [6]:
print("Missing Values in Training Dataset:")
display(train_df.isnull().sum())

print("\nMissing Values in Validation Dataset:")
display(valid_df.isnull().sum())

Missing Values in Training Dataset:


Id              0
Title           0
Body            0
Tags            0
CreationDate    0
Y               0
dtype: int64


Missing Values in Validation Dataset:


Id              0
Title           0
Body            0
Tags            0
CreationDate    0
Y               0
dtype: int64


### 1.5 Duplicate Records Analysis

We check both datasets for duplicate records.

Duplicate records may lead to repeated documents in the retrieval collection, which can affect the quality of search results.


In [7]:
print("Duplicate Rows in Training Dataset:",
      train_df.duplicated().sum())

print("Duplicate Rows in Validation Dataset:",
      valid_df.duplicated().sum())

Duplicate Rows in Training Dataset: 0
Duplicate Rows in Validation Dataset: 0


## 2. Data Cleaning and Preprocessing

### 2.1 Checking Text Content

In this step, we inspect the text fields of the dataset, including question titles, bodies, and tags.

We also examine the distribution of question quality labels to understand the different classes available in the dataset.

In [8]:
# Display sample question titles
print("Sample Titles:")
display(train_df["Title"].head(10))

# Display sample question bodies
print("\nSample Bodies:")
display(train_df["Body"].head(5))

# Display sample tags
print("\nSample Tags:")
display(train_df["Tags"].head(10))

# Check label distribution
print("\nQuality Label Distribution:")
display(train_df["Y"].value_counts())

# Check unique labels
print("\nUnique Labels:")
print(train_df["Y"].unique())

Sample Titles:


0               Java: Repeat Task Every Random Seconds
1                    Why are Java Optionals immutable?
2    Text Overlay Image with Darkened Opacity React...
3           Why ternary operator in swift is so picky?
4                   hide/show fab with scale animation
5    Accessing pointer member of the structure usin...
6    How To Disable 2nd Saturday 4th Saturday ,Sund...
7                     Resizing containers in bootstrap
8    Changing Theme in Windows 10 UWP App Programma...
9                         TextBox Value Disappears VB6
Name: Title, dtype: object


Sample Bodies:


0    <p>I'm already familiar with repeating tasks e...
1    <p>I'd like to understand why Java 8 Optionals...
2    <p>I am attempting to overlay a title over an ...
3    <p>The question is very simple, but I just cou...
4    <p>I'm using custom floatingactionmenu. I need...
Name: Body, dtype: object


Sample Tags:


0                                       <java><repeat>
1                                     <java><optional>
2    <javascript><image><overlay><react-native><opa...
3    <swift><operators><whitespace><ternary-operato...
4    <android><material-design><floating-action-but...
5                       <c><pointers><data-structures>
6                      <javascript><jquery><jquery-ui>
7               <jquery><html><css><twitter-bootstrap>
8    <c#><windows-10><windows-10-mobile><windows-10...
9                                                <vb6>
Name: Tags, dtype: object


Quality Label Distribution:


Y
LQ_CLOSE    15000
HQ          15000
LQ_EDIT     15000
Name: count, dtype: int64


Unique Labels:
['LQ_CLOSE' 'HQ' 'LQ_EDIT']


### 2.2 Inspecting HTML Elements and Code Blocks

Stack Overflow question bodies may contain HTML elements, hyperlinks, images, and code snippets.

In this step, we identify common HTML elements and inspect examples containing code blocks.

This helps us decide how to extract readable text while preserving useful programming information.

In [9]:
# Count common HTML elements in question bodies
html_elements = {
    "Paragraphs": r"<p\b",
    "Code blocks": r"<pre\b",
    "Inline code": r"<code\b",
    "Links": r"<a\b",
    "Images": r"<img\b"
}

print("HTML Elements Count:\n")

for element, pattern in html_elements.items():
    count = train_df["Body"].str.contains(
        pattern,
        case=False,
        na=False,
        regex=True
    ).sum()

    print(f"{element}: {count}")


# Display examples of questions containing code
print("\nExamples of Questions Containing Code:")

code_examples = train_df.loc[
    train_df["Body"].str.contains(
        r"<(?:pre|code)\b",
        case=False,
        na=False,
        regex=True
    ),
    "Body"
].head(3)

for i, body in enumerate(code_examples, start=1):
    print(f"\nExample {i}:")
    print(body[:1500])

HTML Elements Count:

Paragraphs: 30247
Code blocks: 17826
Inline code: 20358
Links: 6634
Images: 1971

Examples of Questions Containing Code:

Example 1:
<p>I am attempting to overlay a title over an image - with the image darkened with a lower opacity. However, the opacity effect is changing the overlaying text as well - making it dim. Any fix to this? Here is what is looks like:</p>

<p><a href="https://i.stack.imgur.com/1HzD7.png" rel="noreferrer"><img src="https://i.stack.imgur.com/1HzD7.png" alt="enter image description here"></a></p>

<p>And here is my code for the custom component (article preview - which the above image is a row of article preview components): </p>

<pre><code>//component for article preview touchable image
/* will require the following
- rss feed and api
- user's keyword interests from parse In home.js
- parse db needs to be augmented to include what they heart
- parse db needs to be augmented to include what they press on (like google news)
*/
var React = re

### 2.3 Removing HTML Tags

Stack Overflow question bodies contain HTML markup that is not necessary for text retrieval.

In this step, we use BeautifulSoup to extract readable text from question bodies while preserving programming code content.

The cleaning process is applied to both training and validation datasets.

In [10]:
import warnings
from bs4 import XMLParsedAsHTMLWarning
warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

def clean_html(text):
    soup = BeautifulSoup(text, "html.parser")
    text = soup.get_text(separator=" ", strip=True)
    return re.sub(r"\s+", " ", text).strip()

train_df["Body"] = train_df["Body"].apply(clean_html)
valid_df["Body"] = valid_df["Body"].apply(clean_html)

### 2.4 Cleaning Question Titles

In this step, we remove unnecessary leading and trailing whitespace from question titles.

This ensures consistent formatting across the dataset.

In [11]:
train_df["Title"] = train_df["Title"].str.strip()
valid_df["Title"] = valid_df["Title"].str.strip()

### 2.5 Cleaning Tags

In this step, we normalize the formatting of question tags by removing unnecessary whitespace.

Tags provide useful information about the programming languages, technologies, and topics related to each question.

In [12]:
def clean_tags(tags):
    return tags.replace("><", " ").replace("<", "").replace(">", "").strip()

train_df["Tags"] = train_df["Tags"].apply(clean_tags)
valid_df["Tags"] = valid_df["Tags"].apply(clean_tags)

### 2.6 Checking the Cleaned Data

After applying the cleaning operations, we inspect sample records to verify that the text fields have been processed correctly.

In [13]:
display(train_df[["Title", "Body", "Tags"]].head(5))

,Title,Body,Tags
0,Java: Repeat Task Every Random Seconds,I'm already familiar with repeating tasks ever...,java repeat
1,Why are Java Optionals immutable?,I'd like to understand why Java 8 Optionals we...,java optional
2,Text Overlay Image with Darkened Opacity React...,I am attempting to overlay a title over an ima...,javascript image overlay react-native opacity
3,Why ternary operator in swift is so picky?,"The question is very simple, but I just could ...",swift operators whitespace ternary-operator op...
4,hide/show fab with scale animation,I'm using custom floatingactionmenu. I need to...,android material-design floating-action-button


### 2.7 Final Data Quality Check

In this step, we perform a final check on the cleaned datasets to verify that there are no missing values or duplicate records after preprocessing.

In [14]:
print("Training Dataset:")
print("Missing Values:", train_df.isnull().sum().sum())
print("Duplicate Rows:", train_df.duplicated().sum())

print("\nValidation Dataset:")
print("Missing Values:", valid_df.isnull().sum().sum())
print("Duplicate Rows:", valid_df.duplicated().sum())

Training Dataset:
Missing Values: 0
Duplicate Rows: 0

Validation Dataset:
Missing Values: 0
Duplicate Rows: 0


In [15]:
# Duplicates based on text only (ignoring Id)
print("Duplicate texts in train:", train_df.duplicated(subset=["Title", "Body"]).sum())
print("Duplicate texts in valid:", valid_df.duplicated(subset=["Title", "Body"]).sum())

# Empty bodies after cleaning
print("Empty bodies in train:", (train_df["Body"].str.strip() == "").sum())
print("Empty bodies in valid:", (valid_df["Body"].str.strip() == "").sum())

Duplicate texts in train: 0
Duplicate texts in valid: 0
Empty bodies in train: 3
Empty bodies in valid: 3


In [16]:
empty_train = train_df[train_df["Body"].str.strip() == ""]
display(empty_train[["Id", "Title", "Tags"]])

,Id,Title,Tags
23032,45343135,Can I dynamically updated the marker location ...,javascript jquery google-maps
26052,46790594,i m write the below jquery first ajax call ret...,javascript php jquery ajax
38976,55026674,How can i update status field as approved ; wh...,php jquery ajax


In [17]:
before = len(train_df)
train_df = train_df[train_df["Body"].str.strip() != ""].reset_index(drop=True)
print("Removed:", before - len(train_df))
print("Train now:", len(train_df))

Removed: 3
Train now: 44997


## 3. Creating documents.csv

In this step, we prepare the document collection for the Information Retrieval system.

Each Stack Overflow question will be treated as a document. We combine the question title, body, and tags into one text field called Document.

In [18]:
# Create a copy of the training dataset
documents_df = train_df.copy()

# Combine Title, Body, and Tags into one Document column
documents_df["Document"] = (
    documents_df["Title"].fillna("") + " " +
    documents_df["Body"].fillna("") + " " +
    documents_df["Tags"].fillna("")
)

# Display the first 5 documents
display(documents_df[["Id", "Document", "Y"]].head())

,Id,Document,Y
0,34552656,Java: Repeat Task Every Random Seconds I'm alr...,LQ_CLOSE
1,34553034,Why are Java Optionals immutable? I'd like to ...,HQ
2,34553174,Text Overlay Image with Darkened Opacity React...,HQ
3,34553318,Why ternary operator in swift is so picky? The...,HQ
4,34553755,hide/show fab with scale animation I'm using c...,HQ


### 3.1 Saving the Document Collection

We save the prepared document collection as a CSV file to use it later in the Information Retrieval system.

In [19]:
# Save the document collection (compressed)
documents_df.to_csv("/kaggle/working/documents.csv.gz", index=False, compression="gzip")

print("documents.csv.gz saved successfully!")
print("Number of documents:", len(documents_df))

# Check file size
import os
size_mb = os.path.getsize("/kaggle/working/documents.csv.gz") / (1024 * 1024)
print(f"File size: {size_mb:.1f} MB")

documents.csv.gz saved successfully!
Number of documents: 44997
File size: 15.2 MB
